# 07 · Learning reusable abstractions
### How repeated program structure becomes a *paid-for*, executable library

In lessons **02** and **06**, we represented programs as trees and contrasted syntactic structure with execution behavior. In **04** and **05**, we asked which candidate to search next. Here we change the **search language itself**: instead of repeatedly rediscovering a useful sequence of primitive operations, we turn that sequence into a callable new primitive.

**Question:** When does a reusable abstraction justify its definition cost, and when does it genuinely help solve previously unseen tasks faster?

Our laboratory moves from **exact repeated subtrees** to **parameterized program macros**, then measures:

1. Whether the macro preserves execution **on every possible input** in a tiny test domain.
2. Whether a library reduces the **combined description length** of its definitions and the training corpus.
3. Whether it improves **held-out search effort** rather than just shortening familiar training programs.
4. Whether adding a new action can also **hurt** search by increasing branching factor.

We build every data structure and algorithm in NumPy/Python. Matplotlib supplies visual demonstrations. No external datasets, GPUs, or ML libraries required.

> This is a **small original pedagogical greedy library learner**, not a reimplementation of DreamCoder or Stitch. It deliberately uses limited candidate templates, a transparent unit-node proxy for description length, and width-seven Boolean rows. The toy exhaustive equivalence tests do **not** transfer to arbitrary ARC grids.

**Reading path:** AST + interpreter → reuse motifs → lambda-like parameterization → capture and expansion → MDL → greedy compression → behavioral proof by exhaustive toy testing → search depth and branch cost → negative transfer → Grammar-UCT integration → exercises.

**Predict before running** each experiment; compare the plotted results to your own hypothesis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from collections import Counter, deque
from itertools import product

SEED = 707
WIDTH = 7
OPS_UNARY = ("RIGHT", "LEFT", "FLIP", "NOT")
OPS_BINARY = ("OR", "AND")
ARITY = {"INPUT": 0, "ZERO": 0, "ARG": 0,
         **{op:1 for op in OPS_UNARY},
         **{op:2 for op in OPS_BINARY}}
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": .2})
print("Reproducible, self-contained toy library-learning environment.")

## 1 · Programs are trees and their leaves are values

Our *base language* processes a seven-cell Boolean row. Unary operators shift right/left, flip order, or invert cells; OR and AND combine two masks. **INPUT** means the current task input; **ZERO** is a constant empty mask.

The syntax below deliberately mirrors lesson 06. A program AST records exact composition order. **ARG** will be used *only inside macro definitions*: it is a placeholder to receive an expression supplied at a call site, not another task input.

A macro definition such as

\[\mathrm{M}(z)=\mathrm{RIGHT}(\mathrm{NOT}(z))\]

is reusable with **different arguments**, e.g. \(M(\mathrm{INPUT})\) and \(M(\mathrm{FLIP}(\mathrm{INPUT}))\). Merely assigning a new name to a **closed** subtree tied to INPUT would not provide the same flexibility.

**Predict:** Will RIGHT(NOT(INPUT)) equal NOT(RIGHT(INPUT)) at the boundary? Why does composition order matter?

In [ ]:
@dataclass(frozen=True)
class Expr:
    op: str
    children: tuple = ()
    def __post_init__(self):
        if any(not isinstance(ch, Expr) for ch in self.children):
            raise TypeError("Children must be expression nodes")
        arity = ARITY.get(self.op, 1 if self.op.startswith("M_") else None)
        if arity is None or len(self.children) != arity:
            raise ValueError(f"Invalid arity or operator {self.op}")

def U(op, child): return Expr(op,(child,))
def B(op, a, b): return Expr(op,(a,b))
X=Expr("INPUT");ZERO=Expr("ZERO");ARG=Expr("ARG")
def size(e): return 1+sum(size(ch) for ch in e.children)
def depth(e): return 0 if not e.children else 1+max(depth(ch) for ch in e.children)
def pretty(e): return e.op if not e.children else e.op+"("+", ".join(map(pretty,e.children))+")"
def iter_subtrees(e):
    yield e
    for ch in e.children: yield from iter_subtrees(ch)

MOTIF=U("RIGHT",U("NOT",ARG))
demo=U("RIGHT",U("NOT",U("FLIP",X)))
print("Abstract template:",pretty(MOTIF))
print("Concrete example:",pretty(demo))
assert size(MOTIF)==3 and size(demo)==4
try:
    B("RIGHT",X,X)
    raise AssertionError("Incorrect arity escaped validation")
except ValueError:
    print("Invalid arity correctly rejected.")

## 2 · See an abstraction as a reusable wiring diagram

The abstract ARG stands for an entire **expression**, not merely a constant. When we call M(FLIP(INPUT)), the complete subtree FLIP(INPUT) replaces the placeholder; no string editing or Python eval is involved.

Our diagrams illustrate an AST. Repeated subtrees can be shared as DAG nodes in production, but *occurrences* must be preserved for accurate cost accounting unless a different DAG-description model is explicitly used.

**Predict:** How many primitive nodes would be evaluated in the expanded expression M(FLIP(INPUT))?

In [ ]:
def plot_ast(ax,e,title):
    locations={};leaf=[0]
    def place(node,path=(),y=0):
        if not node.children:
            x=float(leaf[0]);leaf[0]+=1
        else:
            xx=[place(c,path+(j,),y-1) for j,c in enumerate(node.children)]
            x=float(np.mean(xx))
        locations[path]=(x,y,node)
        return x
    place(e)
    for path,(x,y,node) in locations.items():
        for j,c in enumerate(node.children):
            xx,yy,_=locations[path+(j,)]
            ax.plot([x,xx],[y,yy],color="0.6",lw=1.5)
    for x,y,node in locations.values():
        ax.text(x,y,node.op,ha="center",va="center",fontsize=9,
                bbox=dict(boxstyle="round,pad=.25",fc="#d9e8ff",ec="#6d87ab"))
    ax.set_xlim(-.75,max(.75,leaf[0]-.25))
    ax.set_ylim(-depth(e)-.6,.6)
    ax.set_title(title);ax.axis("off")
fig,axs=plt.subplots(1,3,figsize=(11,4.1),layout="constrained")
plot_ast(axs[0],MOTIF,"Definition: M(ARG)")
plot_ast(axs[1],demo,"Expanded call: M(FLIP(INPUT))")
plot_ast(axs[2],U("M_DEMO",U("FLIP",X)),"Compact call tree")
plt.show()

## 3 · Execute the base language *exactly*

The interpreter accepts either one row of width seven or a batch of rows. RIGHT inserts a false cell at position zero and discards the final original cell; LEFT does the converse. A macro is **expanded before execution** rather than approximated by a network.

This separation is central: abstraction learning may be statistical or heuristic, but calling an admitted macro should have well-defined exact semantics.

**Predict:** Can two expressions be different syntax trees and still produce identical outputs for all inputs?

In [ ]:
def execute(e,rows):
    g=np.asarray(rows,dtype=bool)
    if g.ndim not in (1,2) or g.shape[-1]!=WIDTH: raise ValueError("Expected width-seven Boolean grids")
    if e.op=="INPUT": return g.copy()
    if e.op=="ZERO": return np.zeros_like(g)
    if e.op=="ARG": raise ValueError("ARG is a template placeholder, not executable")
    if e.op.startswith("M_"): raise ValueError("Expand library macros before evaluation")
    children=[execute(ch,g) for ch in e.children]
    if e.op=="RIGHT":
        r=np.zeros_like(children[0]);r[...,1:]=children[0][...,:-1];return r
    if e.op=="LEFT":
        r=np.zeros_like(children[0]);r[...,:-1]=children[0][...,1:];return r
    if e.op=="FLIP": return children[0][...,::-1].copy()
    if e.op=="NOT": return ~children[0]
    if e.op=="OR": return children[0]|children[1]
    if e.op=="AND": return children[0]&children[1]
    raise ValueError(e.op)
ALL_INPUTS=((np.arange(2**WIDTH,dtype=np.uint16)[:,None]>>np.arange(WIDTH)) & 1).astype(bool)
assert ALL_INPUTS.shape==(128,WIDTH)
def signature(e): return execute(e,ALL_INPUTS).astype(np.uint8).tobytes()
def semantically_equal(a,b): return signature(a)==signature(b)
assert semantically_equal(U("FLIP",U("FLIP",X)),X)
assert not semantically_equal(U("RIGHT",U("NOT",X)),U("NOT",U("RIGHT",X)))
sample=ALL_INPUTS[[1,14,37,91]]
fig,axs=plt.subplots(1,3,figsize=(11,2.9),layout="constrained")
for ax,expr,title in zip(axs,[X,U("RIGHT",U("NOT",X)),U("NOT",U("RIGHT",X))],
                         ["Input","RIGHT after NOT","NOT after RIGHT"]):
    ax.imshow(execute(expr,sample),cmap="Greys",vmin=0,vmax=1,aspect="auto")
    ax.set(xticks=range(WIDTH),yticks=range(len(sample)),xlabel="Cell",title=title)
plt.show()

## 4 · Generate a *training-only* corpus with recurring structure

In a learned library, candidate abstractions should be proposed from programs **already solved on training tasks**. Here we construct a transparent toy corpus whose solutions contain a reusable family \(R(N(z))\) with several different arguments \(z\). We also include distractor motifs and unrelated operations.

This corpus is **synthetically engineered** to demonstrate when library learning works; it does not establish that this motif occurs frequently in ARC. In particular, it would be invalid to discover a library on hidden evaluation programs and then report those same programs as held-out search improvements.

**Predict:** If R(N(x)) recurs with different child expressions, is its most useful abstraction a full concrete subtree or a parameterized context?

In [ ]:
BASE_ARGS=[X,U("FLIP",X),U("LEFT",X),U("RIGHT",X),
           U("NOT",X),B("OR",X,U("LEFT",X)),B("AND",U("RIGHT",X),X)]
def R_N(p): return U("RIGHT",U("NOT",p))
def L_N(p): return U("LEFT",U("NOT",p))
train_corpus=[]
for z in BASE_ARGS:
    r=R_N(z);l=L_N(z)
    train_corpus += [r,U("FLIP",r),U("NOT",r),B("OR",r,U("LEFT",z)),
                     B("AND",r,U("FLIP",z)),
                     l,B("OR",l,U("FLIP",z)),
                     U("FLIP",U("LEFT",z))]
train_corpus=list(dict.fromkeys(train_corpus))
print("Training programs:",len(train_corpus))
print("Distinct concrete R(N(arg)) ASTs:",
      len({R_N(z) for z in BASE_ARGS}))
print("Examples of shared context:")
for p in train_corpus[:5]:print("  ",pretty(p))
assert len(train_corpus)>40
assert len({R_N(z) for z in BASE_ARGS})==len(BASE_ARGS)

## 5 · Template matching is a small form of anti-unification

A template such as RIGHT(NOT(ARG)) matches any AST with the same two outer operators; ARG binds the entire remaining child tree. It is a one-variable first-order **context matcher**, not full higher-order anti-unification. With multiple occurrences of ARG, they must bind to the **same AST**, preventing incorrect reuse.

Examples:
- RIGHT(NOT(INPUT)) matches M(ARG) with ARG = INPUT.
- RIGHT(NOT(FLIP(INPUT))) matches with ARG = FLIP(INPUT).
- RIGHT(LEFT(INPUT)) does **not** match.
- OR(ARG,ARG) matches OR(INPUT,INPUT) but not OR(INPUT,ZERO).

**Predict:** Why should the same ARG placeholder never match two different child expressions in one macro call?

In [ ]:
def match_template(pattern,subject,binding=None):
    if binding is None: binding={}
    if pattern.op=="ARG":
        if "ARG" not in binding:
            return {"ARG":subject}
        return binding if binding["ARG"]==subject else None
    if pattern.op!=subject.op or len(pattern.children)!=len(subject.children):return None
    for p,s in zip(pattern.children,subject.children):
        binding=match_template(p,s,binding)
        if binding is None:return None
    return binding

assert match_template(MOTIF,R_N(X))["ARG"]==X
assert match_template(MOTIF,R_N(U("FLIP",X)))["ARG"]==U("FLIP",X)
assert match_template(MOTIF,U("RIGHT",U("LEFT",X))) is None
repeated=B("OR",ARG,ARG)
assert match_template(repeated,B("OR",X,X)) is not None
assert match_template(repeated,B("OR",X,ZERO)) is None
fig,ax=plt.subplots(figsize=(9,3.4))
cases=[R_N(X),R_N(U("FLIP",X)),U("RIGHT",U("LEFT",X)),R_N(U("NOT",X))]
colors=["#83c79e" if match_template(MOTIF,p) is not None else "#e6a1a1" for p in cases]
ax.barh(range(len(cases)),[size(p) for p in cases],color=colors)
ax.set(yticks=range(len(cases)),yticklabels=[pretty(p) for p in cases],
       xlabel="AST node count",title="Which programs match RIGHT(NOT(ARG))?")
ax.invert_yaxis();plt.show()

## 6 · Capture a pattern and call it as an operation

Library entry **M_0(ARG) := RIGHT(NOT(ARG))** has two responsibilities:

- **Compression:** find matching AST contexts and replace each with a node M_0(child).
- **Execution:** expand every M_0(child) back into RIGHT(NOT(child)) before invoking the trusted interpreter.

A rewrite should recursively process a matched argument: repeated nested motifs can then be compressed twice. A substitution must **replace whole AST nodes**, never manipulate printed source strings. We check that expansion is semantics-preserving.

**Predict:** If R(N(R(N(INPUT))) is compressed, how many M_0 calls should remain?

In [ ]:
@dataclass(frozen=True)
class Macro:
    name: str
    body: Expr
    def __post_init__(self):
        if not self.name.startswith("M_"): raise ValueError("Macro names must start with M_")
        if "ARG" not in [node.op for node in iter_subtrees(self.body)]:
            raise ValueError("A reusable unary macro must use ARG")

def substitute(pattern,arg):
    return arg if pattern.op=="ARG" else Expr(pattern.op,
                          tuple(substitute(ch,arg) for ch in pattern.children))

def expand(e,library,stack=()):
    if e.op=="ARG": raise ValueError("Unbound template ARG")
    if e.op in library:
        if e.op in stack:raise ValueError("Cyclic macro definition")
        macro=library[e.op]
        # Nested calls M(M(x)) are legal; expand the argument outside the
        # definition-dependency stack. Only references inside a macro body can cycle.
        argument=expand(e.children[0],library,stack)
        replaced=substitute(macro.body,argument)
        return expand(replaced,library,stack+(e.op,))
    return Expr(e.op,tuple(expand(ch,library,stack) for ch in e.children))

def rewrite_one(e,macro):
    binding=match_template(macro.body,e)
    if binding is not None:
        return U(macro.name,rewrite_one(binding["ARG"],macro))
    return Expr(e.op,tuple(rewrite_one(ch,macro) for ch in e.children))

def rewrite_corpus(corpus,macros):
    out=list(corpus)
    for macro in macros:
        out=[rewrite_one(e,macro) for e in out]
    return out

macro=Macro("M_0",MOTIF)
nested=R_N(R_N(X))
compressed=rewrite_one(nested,macro)
print("Original:",pretty(nested))
print("Compressed:",pretty(compressed))
print("Expanded:",pretty(expand(compressed,{macro.name:macro})))
assert pretty(compressed)=="M_0(M_0(INPUT))"
assert expand(compressed,{macro.name:macro})==nested
assert size(compressed)==3 and size(nested)==5

## 7 · One AST can be shorter while its expanded semantics stay identical

All 128 Boolean input rows are available, so we can exhaustively verify a macro's behavior for **this finite toy domain**. We check the original, compressed, and expanded program variants—not a neural prediction.

The transformation is actually justified by **structural substitution**, not the 128-input test; exhaustive checking here guards our implementation. On general infinite domains or unbounded grids, finite sampled tests would not prove semantic identity.

**Predict:** Is equivalence on the training input examples sufficient to treat two ASTs as universally equivalent?

In [ ]:
library_one={macro.name:macro}
rewritten=rewrite_corpus(train_corpus,[macro])
for original,short in zip(train_corpus,rewritten):
    assert expand(short,library_one)==original
    assert np.array_equal(execute(original,ALL_INPUTS),
                          execute(expand(short,library_one),ALL_INPUTS))
sizes_before=np.array([size(p) for p in train_corpus])
sizes_after=np.array([size(p) for p in rewritten])
fig,ax=plt.subplots(figsize=(9,3.6))
ax.scatter(sizes_before,sizes_after,alpha=.67,s=30)
lims=(0,max(sizes_before.max(),sizes_after.max())+1)
ax.plot(lims,lims,ls="--",color="gray",label="No compression")
ax.set(xlim=lims,ylim=lims,xlabel="Original AST nodes",ylabel="AST nodes with M_0 calls",
       title="Exact rewrite reduces length on repeated contexts")
ax.legend();plt.show()
print("All",len(train_corpus),"training programs preserve exact AST expansion and semantics.")

## 8 · Candidate discovery: search **contexts**, not only exact subtrees

An exact subtree frequency count only finds identical concrete expressions. A reusable unary macro can capture **different child subtrees** through ARG. In this introductory learner, candidate generation scans every two-step **unary chain**:

\[\mathrm{OUTER}(\mathrm{INNER}(z)),\quad z\;\text{arbitrary expression}.\]

This is deliberately a small fragment of library-learning search. We do not enumerate higher-order abstractions, arbitrary multi-argument templates, recursion, or e-graph equivalence classes.

**Predict:** Is the most frequent two-step chain necessarily the best macro under a description-length objective?

In [ ]:
def unary_chain_templates(corpus):
    result=set()
    for e in corpus:
        for sub in iter_subtrees(e):
            if sub.op in OPS_UNARY and sub.children[0].op in OPS_UNARY:
                result.add(U(sub.op,U(sub.children[0].op,ARG)))
    return sorted(result,key=pretty)

def count_occurrences(corpus,template):
    return sum(match_template(template,sub) is not None
               for e in corpus for sub in iter_subtrees(e))

templates=unary_chain_templates(train_corpus)
occurrences=np.array([count_occurrences(train_corpus,t) for t in templates])
order=np.argsort(-occurrences)
fig,ax=plt.subplots(figsize=(9.7,3.7))
shown=order[:10]
ax.bar(range(len(shown)),occurrences[shown])
ax.set(xticks=range(len(shown)),
       xticklabels=[pretty(templates[i]).replace("(ARG)","(z)") for i in shown],
       ylabel="Raw syntactic occurrences",title="Candidate contexts in solved training programs")
ax.tick_params(axis="x",rotation=60,labelsize=8);plt.show()
print("Distinct unary-chain templates:",len(templates))
assert MOTIF in templates and count_occurrences(train_corpus,MOTIF)>len(BASE_ARGS)

## 9 · Minimum description length must pay for the library

A macro that shortens one program may still be a bad addition after accounting for its **definition**, symbol table, and call overhead.

We use an explicit *node-count proxy*:

\[L(C,\mathcal L)=\underbrace{\sum_{p\in C} \#\text{AST nodes in rewritten }p}_{\text{compressed corpus}}
+\underbrace{\sum_{m\in\mathcal L}(\#\text{nodes in definition}_m+2)}_{\text{definitions + overhead}}.\]

**This is not a complete prefix-free code or exact bit-length MDL.** It uses unit costs to reveal amortization. A production description-length model would also encode symbol IDs, arities, parameters, counts, type schemes, variable bindings, and any downstream learned parameters.

For this two-operator macro, replacing two nested unary operations with one macro call saves **one node per nonoverlapping use**; the definition costs 3 nodes + a 2-node overhead = 5. It breaks even after five uses, and strictly improves after six.

**Predict:** How many calls are needed if a new macro is used only once per task?

In [ ]:
DEF_OVERHEAD=2
def macro_cost(m): return size(m.body)+DEF_OVERHEAD
def description_length(corpus,macros):
    return sum(size(p) for p in rewrite_corpus(corpus,macros)) + sum(macro_cost(m) for m in macros)

use_counts=np.arange(0,17)
net_savings=use_counts*(size(MOTIF)-2)-macro_cost(macro)
fig,ax=plt.subplots(figsize=(8.8,3.5))
ax.bar(use_counts,net_savings)
ax.axhline(0,color="gray",lw=1)
ax.set(xticks=use_counts[::2],xlabel="Nonoverlapping uses",ylabel="Total description-length saving",
       title="The definition must amortize across multiple call sites")
plt.show()
print("Definition cost:",macro_cost(macro),
      "| one-use savings:",size(MOTIF)-2,
      "| first profitable use count:",next(i for i,v in zip(use_counts,net_savings) if v>0))
assert macro_cost(macro)==5
assert description_length(train_corpus,[])==sum(size(p) for p in train_corpus)

## 10 · Greedily learn a library from the training corpus

We propose all two-unary-step contexts, ask which one lowers the current total description length the most, add it as a macro, rewrite the corpus, and repeat until no proposed candidate improves the objective.

**Training only:** the macro learner sees **train_corpus**, never held-out synthesis targets. Its outputs are not guaranteed to be a globally optimal library. Local greed can miss useful combinations, and different templates can overlap.

We freeze macro definitions at their original primitive expression. This keeps dependencies acyclic and expansion easy to verify. More advanced systems can learn *nested* library dependencies and jointly re-optimize the grammar.

**Predict:** Why might the best second macro differ from the second-highest frequency pattern in the original corpus?

In [ ]:
def learn_greedy_library(corpus,candidates,maximum=5):
    selected=[];remaining=list(candidates)
    trace=[description_length(corpus,selected)]
    report=[]
    for i in range(maximum):
        baseline=description_length(corpus,selected)
        options=[]
        for template in remaining:
            trial=Macro(f"M_{len(selected)}",template)
            new_score=description_length(corpus,selected+[trial])
            options.append((baseline-new_score,pretty(template),template,new_score))
        if not options:break
        gain,_,best,new_score=max(options,key=lambda row:(row[0],row[1]))
        if gain<=0:break
        selected.append(Macro(f"M_{len(selected)}",best))
        remaining.remove(best)
        trace.append(new_score)
        report.append((selected[-1].name,pretty(best),gain,new_score))
    return selected,np.array(trace),report

selected,trace,report=learn_greedy_library(train_corpus,templates,maximum=5)
for name,body,gain,new_score in report:
    print(f"{name} := {body:27s} | gain={gain:4d} | total={new_score}")
print("Training-only objective: initial",int(trace[0]),"final",int(trace[-1]))
fig,ax=plt.subplots(figsize=(9,3.6))
ax.plot(range(len(trace)),trace,marker="o",lw=2)
ax.set(xticks=range(len(trace)),xlabel="Accepted library macros",ylabel="Corpus + library node cost",
       title="Greedy compression objective on training programs")
plt.show()
assert len(selected)>0
assert np.all(np.diff(trace)<0)
assert trace[-1]<trace[0]

## 11 · Verify the complete learned library

A library is useful only if its calls can be expanded and executed with no loss of semantics. After greedy selection, we verify each compressed program expands **exactly** into its original AST. This is stronger than merely matching the training examples.

We also test every rewritten program on the exhaustive 128-input Boolean universe, checking the execution engine end to end. Again, exhaustive verification is feasible solely because of our unusually tiny domain.

**Predict:** Would swapping the order of library rewrites necessarily leave corpus cost unchanged when patterns overlap?

In [ ]:
lib={m.name:m for m in selected}
short_corpus=rewrite_corpus(train_corpus,selected)
for original,short in zip(train_corpus,short_corpus):
    expanded=expand(short,lib)
    assert expanded==original
    assert np.array_equal(execute(original,ALL_INPUTS),execute(expanded,ALL_INPUTS))
assert description_length(train_corpus,selected)==trace[-1]
root_before=sum(size(p) for p in train_corpus)
root_after=sum(size(p) for p in short_corpus)
library_bill=sum(macro_cost(m) for m in selected)
fig,ax=plt.subplots(figsize=(8.8,3.5))
ax.bar(["Primitive corpus","Rewritten corpus","Definition cost"],
       [root_before,root_after,library_bill])
ax.set(ylabel="Unit-node description cost",
       title="Report library overhead separately from code savings")
plt.show()
print("Original program nodes:",root_before,
      "| rewritten program nodes:",root_after,
      "| library definition bill:",library_bill)

## 12 · Structural reuse, semantics, and true abstraction are different

A macro is defined by a *syntactic context* with an argument. It can preserve behavior by construction even if it has no intrinsic semantic interpretation beyond "execute these operations." Conversely, two different programs can be behaviorally equivalent and still have different descriptions:

\[\mathrm{FLIP}(\mathrm{FLIP}(x))=x,\quad
\mathrm{OR}(x,\mathrm{ZERO})=x.\]

Our candidate generator does not exploit these algebraic identities. Systems with equivalence-aware search (e.g., e-graphs or verified rewriting) can identify shared structure **modulo transformations**, but they must guard the applicability of each identity.

**Predict:** Can a corpus with many redundant programs appear highly compressible without containing many genuinely useful new concepts?

In [ ]:
equivalent_pairs=[
    (X,U("FLIP",U("FLIP",X))),
    (X,B("OR",X,ZERO)),
    (R_N(X),U("FLIP",U("FLIP",R_N(X))))
]
fig,axs=plt.subplots(1,3,figsize=(11,3),layout="constrained")
probe=ALL_INPUTS[[1,7,42,103]]
for ax,(a,b) in zip(axs,equivalent_pairs):
    agreement=float(np.mean(execute(a,ALL_INPUTS)==execute(b,ALL_INPUTS)))
    ax.bar(["A","B"],[size(a),size(b)])
    ax.set(title=f"Cell agreement={agreement:.2f}",ylabel="AST size")
plt.show()
assert semantically_equal(equivalent_pairs[0][0],equivalent_pairs[0][1])
assert semantically_equal(equivalent_pairs[1][0],equivalent_pairs[1][1])

## 13 · A library changes both program *depth* and action branching

The search grammar now contains additional macro actions. A useful compound operation can make a formerly deep expression shallow—**at the price of increasing the number of legal next actions**.

To quantify this without hidden implementation tricks, define a toy search process that starts from INPUT and repeatedly wraps its current expression in *one unary operator*. Breadth-first search tries all candidate unary-action sequences of shorter length first.

- **Primitive grammar:** RIGHT, LEFT, FLIP, NOT.
- **Macro grammar:** the same four actions plus one *training-learned* macro.

We judge a candidate by its **full function on all 128 inputs**. That serves as an exact oracle in this controlled domain, not as a realistic ARC verification method. We count each terminal expression executed, including functionally duplicate programs.

**Predict:** Does fewer macro steps automatically imply fewer evaluations? Why not?

In [ ]:
def search_unary_target(target,actions,lib,budget=20000,max_depth=7):
    desired=execute(expand(target,lib),ALL_INPUTS)
    queue=deque([X])
    evaluated=0;seen_full_functions=set()
    while queue and evaluated<budget:
        p=queue.popleft()
        result=execute(expand(p,lib),ALL_INPUTS)
        evaluated+=1
        seen_full_functions.add(result.astype(np.uint8).tobytes())
        if np.array_equal(result,desired):
            return dict(found=True,executions=evaluated,steps=depth(p),program=p,
                        distinct_functions=len(seen_full_functions))
        if depth(p)<max_depth:
            for action in actions:
                queue.append(U(action,p))
    return dict(found=False,executions=evaluated,steps=None,program=None,
                distinct_functions=len(seen_full_functions))

# Pick the single discovered training macro whose body is the intended R(N(ARG)) context.
matching=[m for m in selected if m.body==MOTIF]
assert matching, "Expected the engineered R(N(ARG)) training motif to be learned"
use_macro=matching[0]
demo_library={use_macro.name:use_macro}
primitive_actions=OPS_UNARY
macro_actions=OPS_UNARY+(use_macro.name,)
targets=[R_N(R_N(X)),R_N(R_N(R_N(X))),
         U("NOT",U("LEFT",U("FLIP",X)))]
search_results=[]
for goal in targets:
    baseline=search_unary_target(goal,primitive_actions,{},max_depth=7)
    with_macro=search_unary_target(goal,macro_actions,demo_library,max_depth=7)
    search_results.append((baseline,with_macro))
    print(pretty(goal))
    print("  primitives:",baseline["steps"],"steps,",baseline["executions"],"executions")
    print("  with macro:",with_macro["steps"],"steps,",with_macro["executions"],"executions")
assert all(a["found"] and b["found"] for a,b in search_results)

## 14 · Visualize search effort *including negative transfer*

We plot both the **number of individual operations in the discovered expression** and the **number of exact candidate executions** spent to reach each target. A macro can reduce steps while increasing the branching factor. A target unrelated to the new abstraction may become more expensive to discover.

The held-out targets below were **not included in the macro learner's input corpus**. The learner's corpus intentionally excludes the nested motif repetitions used for the evaluation targets. They are constructed from the same known toy domain to test whether the training-learned abstraction transfers to new *compositions*, not whether an unseen real-world domain has been learned.

**Predict:** If the macro is inserted earlier in enumeration order rather than appended last, which numbers might change, and which semantics should not?

In [ ]:
names=["motif×2","motif×3","unrelated"]
fig,axs=plt.subplots(1,2,figsize=(11,3.7),layout="constrained")
v0=[r[0]["steps"] for r in search_results]
v1=[r[1]["steps"] for r in search_results]
e0=[r[0]["executions"] for r in search_results]
e1=[r[1]["executions"] for r in search_results]
x=np.arange(len(names))
axs[0].bar(x-.18,v0,width=.36,label="Base grammar")
axs[0].bar(x+.18,v1,width=.36,label="Base + library")
axs[0].set(xticks=x,xticklabels=names,ylabel="Search steps",title="Shortest found expression depth")
axs[1].bar(x-.18,e0,width=.36,label="Base grammar")
axs[1].bar(x+.18,e1,width=.36,label="Base + library")
axs[1].set(xticks=x,xticklabels=names,yscale="log",ylabel="Exact candidate executions",
            title="Branching cost is part of the budget")
for ax in axs:ax.legend(fontsize=8)
plt.show()
print("Depth-one action counts:",len(primitive_actions),"→",len(macro_actions))
assert e1[0]<e0[0] and e1[1]<e0[1]

## 15 · Library compression is not the same as a good search policy

The minimum-description-length objective rewards fewer *symbolic code units*. A search policy instead cares about the expected return from **choosing an action now** under a limited compute budget.

Compression and search can reinforce each other—**if useful motifs recur on future tasks**—but the following links are not automatic:

- shorter code does not imply easier verification;
- a frequently used macro need not be the best macro on a new task;
- actions added to the grammar compete with existing actions for UCT/PUCT exploration;
- the grammar may contain many redundant or equivalent macros;
- a macro can move an impossible goal into reach at fixed search depth *or* increase search effort for an unrelated goal.

The experiments above provide a concrete version of **negative transfer**. Test macro usefulness on **new tasks**, not only the same training corpus used to propose it.

## 16 · DreamCoder's larger idea: search and abstraction can bootstrap each other

DreamCoder-style library learning connects three ingredients:

1. **Search (wake):** solve current tasks with the grammar and current recognition guidance.
2. **Abstraction/compression:** discover reusable pieces among solved programs and update the language.
3. **Recognition learning (sleep):** train a model to guide subsequent search in the updated language, sometimes using generated tasks to provide extra supervision.

Iterating can create a feedback loop: better search discovers more solution structures; better abstractions shorten future programs; better learned priors exploit the changed grammar. But feedback can also amplify search bias: frequent patterns among *found programs* are not necessarily frequent among *all solutions*.

Our toy notebook implements only **one transparent compression stage** plus a held-out search check. It does not implement DreamCoder's probabilistic grammar, recognition network, wake-sleep training, or e-graph refactoring.

**Predict:** If a learned prior from lesson 05 still emits probabilities only for the old primitive operation set, what must be changed before the new macros are legal search actions?

## 17 · Safe interface to your Grammar-UCT and bundled genes

For your symbolic-regression/ARC search engine, a reusable abstraction should have more than a name. A well-defined candidate record could store:

| Field | Purpose |
|---|---|
| Macro ID + immutable version | Reproducible dependency and cache key |
| Typed parameter and return signatures | Filter illegal source/component combinations |
| Expansion body as AST | Exact, auditable semantics |
| Parameter binding rules | Avoid capture, aliasing, and incorrect substitution |
| Total definition cost | Penalize library growth |
| Training-corpus usage and distinct-task support | Separate local repetition from broad reuse |
| Semantic regression fixtures | Detect accidental changes to old solved programs |
| Measured latency / branching effects | Evaluate search cost rather than code size alone |
| Provenance and source-component path | Preserve one-slot gene identity and nested component identity |

**Crucial distinction:** a macro invocation may produce a *single physical bundled gene* while retaining nested semantic components. A macro's AST, its expansion tree, the generating gene ID, and its component paths are different entities. The 06 embeddings/caches should be invalidated or versioned when the grammar changes.

A staged deployment can evaluate new macros in shadow mode: keep the original grammar as baseline, propose macro candidates on training tasks, then admit only verified abstractions that pass held-out semantic and search-budget tests.

## 18 · Experimental design and threats to validity

In a real assessment, keep the following separate:

1. **Training tasks:** used to synthesize programs and propose macros.
2. **Validation tasks:** used to choose overhead penalties, grammar size, candidate-generation heuristics, or search hyperparameters.
3. **Final test tasks:** held out from all model, library, and hyperparameter tuning.

Compare baselines under **matched execution counts**, **accepted unique gene counts**, and **wall time**. Report exact solved tasks, time to first verified solve, library overhead, macro usage by distinct task, and failures caused by grammar expansion. Use multiple seeds and uncertainty intervals.

The toy universe contains only 128 Boolean inputs at width seven. Exhaustive equivalence and reproducible search results here do not imply robust ARC generalization. As in 03, inferred confidence over *observed solutions* cannot quantify programs you never explored.

## 19 · Exercises: make the abstraction learner less naive

1. **Amortization.** For a context of five nodes with one ARG and a macro-call cost of two nodes, derive the minimum use count that offsets a seven-node definition bill.
2. **Template matching.** Extend to two named placeholders ARG0/ARG1. Explain how shared occurrences constrain bindings.
3. **Semantics.** Why is substituting a whole immutable AST safer than text replacement? Under what extension to a language could variable capture become important?
4. **Parameterization.** Discover a reusable context appearing with at least three distinct ARG bindings; quantify the difference from an exact-concrete-subtree counter.
5. **Nested abstractions.** Allow a newly learned definition to call a previous macro. Add cycle detection and charge the nested definition appropriately.
6. **Overlapping matches.** Construct a corpus where candidate order affects rewriting and cost. How could joint refactoring improve it?
7. **Behavioral equivalence.** Add a verified rewrite FLIP(FLIP(x)) → x. Show its effect on corpus cost before macro learning.
8. **Held-out search.** Add a target with no training motif. Does the enlarged grammar perform better, worse, or the same at matched execution count?
9. **Cost-aware selection.** Make the reward proportional to exact solved tasks per actual execution millisecond. What biases arise from caching and variable input shapes?
10. **Search policy adaptation.** Propose an action-probability extension for lesson 05's network that includes the new macro with an initial prior and permits continued exploration.

Attempt at least three modifications before reading the solutions.

## 20 · Worked solutions and reconstruction checklist

1. Each use saves \(5-2=3\) nodes. Seven definition nodes require at least three uses for strictly positive net saving (3×3−7=2).
2. Bind each named placeholder to its first matching subtree, require structural equality on repeated occurrences, and treat different placeholder names independently.
3. Structural substitution preserves program boundaries and typed nodes. With lexical binders (lambdas, let-bindings), use alpha-renaming or de Bruijn indices to prevent variable capture.
4. For RIGHT(NOT(ARG)), the seven base argument expressions demonstrate reuse across nonidentical concrete ASTs. A single exact subtree has fewer matches.
5. Expand dependencies recursively, detect cycles by an active definition stack, and price both dependencies and calls. Verify the expansion on a finite suite and by structural equality where possible.
6. A macro may swallow an inner occurrence that another candidate would compress. Joint refactoring can optimize the library and all rewritten programs together rather than greedily fixing one at a time.
7. Algebraic canonicalization can reduce both search and compression cost, but a domain-wide law must be verified under exact operation semantics, including boundaries and types.
8. Adding a branch generally increases enumeration work for unrelated targets; at a fixed depth its exact effect depends on action order and solution equivalences.
9. Execution costs vary, repeated programs may be cache hits, and runtime measurements are noisy. Define whether the objective is throughput, latency to first solve, or budgeted solve rate.
10. Extend legal-action embeddings with macro structure, initialize and calibrate its prior on separate training tasks, renormalize over legal actions, and retain a nonzero exploration mechanism.

**Rebuild without looking:** typed AST; interpreter; parameterized matcher; macro expansion; nonoverlapping rewrite; library-inclusive description-length objective; greedy macro discovery; exact behavioral checks; held-out search experiment.

### Primary reading
- Ellis et al. (2021), *DreamCoder: Bootstrapping Inductive Program Synthesis with Wake-Sleep Library Learning*, PLDI. https://doi.org/10.1145/3453483.3454080
- Bowers et al. (2023), *Stitch: Fast Library Learning for Neurosymbolic Program Synthesis* (related library-abstraction discovery approach; consult the actual paper for details).
- Muggleton (1991), *Inductive Logic Programming* (background on learned reusable logical patterns).
- Rissanen (1978), *Modeling by Shortest Data Description* (MDL historical foundation).

**Next: 08 · Adaptation and compositional generalization.** We will hold out new compositions, measure transfer from one task family to another, and test failure modes that a familiar-corpus compression score cannot reveal.

**Scope:** original didactic learner; no claims of state-of-the-art search performance or complete MDL correctness.